In [3]:
import pandas as pd
import sqlite3

1. Use the sqlite3 library to create a connection to the database.

In [4]:
db_file = '../data/checking-logs.sqlite'
conn = sqlite3.connect(db_file)

2. Using one query per group, create two dataframes, test_results and control_results, with the columns "time" and "avg_diff" and two rows.
- The "time" column should contain the values "after" and "before".
- The "avg_diff" should contain the average delta for all users for the time period before and after their first visit to the page.
- Only take into account users with observations before and after.

In [10]:
test_result_query = """
WITH user_period_avg AS (
    SELECT 
        t.uid,
        CASE WHEN t.first_commit_ts < t.first_view_ts THEN 'before' ELSE 'after' END AS period,
        AVG( (d.deadlines - strftime('%s', t.first_commit_ts)) / 3600.0 ) AS avg_delta
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
    GROUP BY t.uid, period
),
users_with_both AS (
    SELECT uid
    FROM user_period_avg
    GROUP BY uid
    HAVING COUNT(DISTINCT period) = 2
)
SELECT 
    period AS time,
    AVG(avg_delta) AS avg_diff
FROM user_period_avg
WHERE uid IN (SELECT uid FROM users_with_both)
GROUP BY period
ORDER BY period;
"""

In [11]:
pd.read_sql(test_result_query , conn)

,time,avg_diff
0,after,100.178181
1,before,66.679583


In [24]:
query = """
WITH user_period_avg AS (
    SELECT 
        c.uid,
        CASE WHEN c.first_commit_ts < c.first_view_ts THEN 'before' ELSE 'after' END AS period,
        AVG( (d.deadlines - strftime('%s', c.first_commit_ts)) / 3600.0 ) AS avg_delta
    FROM control c
    JOIN deadlines d ON c.labname = d.labs
    WHERE c.labname != 'project1'
    GROUP BY c.uid, period
),
users_with_both AS (
    SELECT uid
    FROM user_period_avg
    GROUP BY uid
    HAVING COUNT(DISTINCT period) = 2
)
SELECT 
    period AS time,
    AVG(avg_delta) AS avg_diff
FROM user_period_avg
WHERE uid IN (SELECT uid FROM users_with_both)
GROUP BY period
ORDER BY period;
"""

In [25]:
pd.read_sql(query , conn)

,time,avg_diff
0,after,104.968118
1,before,122.686378


In [26]:
conn.close()

Did the hypothesis turn out to be true, and does the page affect students' behavior?

In the control group (users with an imputed first view), the average delta decreased from 122.69 hours before to 104.97 hours after the imputed date – meaning commits after that point were closer to the deadline.

In [27]:
print("Yes")

Yes
